# 12 — Oriented Ceiling, Buffer Acquisition, and the Ceiling-Gated Selector

Three additions to the four-corpus study on the same caps, splits and seeds as notebooks 10 and 11.

Oriented ceiling. Notebook 11 found 17 of 36 (pair, model) cells where the transferred ranking is inverted: a decreasing transform of the source scores discriminates better than any increasing one. The ceiling is therefore taken over both orientations, and rethreshold_2s chooses orientation and threshold jointly on the buffer. The buffer-side estimate of that two-sided ceiling is recorded as buffer_est.

Acquisition. The random stratified buffers of notebook 10 are compared with four rules for choosing which pool flows to label at the 0.01% and 0.1% budgets: uniform (random from the pool), uncertainty (top predictive entropy under the frozen source model), diversity (k-means with k equal to the budget on standardised features, member nearest each centroid), and hybrid (diversity within the top 5x-budget uncertainty set). Each acquired buffer trains a buffer-only model. The pool is a 200k stratified sample of the target training partition. Uniform and diversity depend only on the target and are recorded once per target under source 'any'; uncertainty and hybrid depend on the source model.

Selector. For each stratified buffer, the buffer-side two-sided ceiling is compared with a 3-fold cross-validated buffer-only MCC on the same buffer; the selector chooses rethreshold when the ceiling estimate is higher and retrain otherwise. Regret is measured against the better realised outcome. Results append to fc_results_v3.csv with resume-skip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(
    corpora       = ['nf2018v2', 'nfunswv2', 'nftonv2', 'nfbotv2'],
    seeds         = [42, 43, 44],
    test_size     = 0.30,
    train_cap     = 250_000,
    eval_cap      = 200_000,
    pool_cap      = 200_000,
    budgets       = [0.0001, 0.001, 0.01, 0.05, 0.10],
    acq_budgets   = [0.0001, 0.001],
    rf_estimators = 300,
    ece_bins      = 15,
    mlp_hidden    = (128, 64),
    mlp_max_iter  = 100,
    cv_folds      = 3,
)
MODELS = ['rf', 'lgbm', 'mlp']
V3_CSV = f'{RESULT}/fc_results_v3.csv'
COLS = ['seed', 'source', 'target', 'model', 'budget', 'strategy', 'n_train',
        'macro_f1', 'weighted_f1', 'mcc', 'auprc_macro', 'fp_rate', 'brier', 'ece',
        'mcc_best_thr', 'thr_best', 'orient', 'buffer_est', 'cv_est', 'n_benign_buf', 'fit_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label', 'Attack')]
for tag, d in DATASETS.items():
    assert [c for c in d.columns if c not in ('Label', 'Attack')] == FEATURES, tag
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    y = np.asarray(y_true).astype(np.int64); p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort'); ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    mccs = mcc_from_counts(tp, fp, P - tp, N - fp)
    i = int(np.argmax(mccs))
    return (0.0, float('inf')) if mccs[i] <= 0 else (float(mccs[i]), float(cuts[i]))

def best_threshold_two_sided(y_true, p_pos):
    """Max MCC over both orientations. Returns (mcc, thr, orient) with orient +1 (p>=thr) or -1 ((1-p)>=thr)."""
    m_pos, t_pos = best_threshold_mcc(y_true, p_pos)
    m_neg, t_neg = best_threshold_mcc(y_true, 1.0 - np.asarray(p_pos))
    return (m_pos, t_pos, 1) if m_pos >= m_neg else (m_neg, t_neg, -1)

def apply_oriented(p_pos, thr, orient):
    p = np.asarray(p_pos)
    return ((p if orient == 1 else 1.0 - p) >= thr).astype(int)

def mcc_of_pred(y_true, pred):
    return matthews_corrcoef(np.asarray(y_true), np.asarray(pred))

# ---------- acquisition rules: return integer positions into the pool ----------
def acquire_uniform(n_pool, k, seed):
    return np.random.default_rng(seed).choice(n_pool, size=min(k, n_pool), replace=False)

def acquire_uncertainty(p_pool, k):
    p = np.clip(np.asarray(p_pool), 1e-9, 1 - 1e-9)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-ent, kind='mergesort')[:k]

def acquire_diversity(X_pool, k, seed):
    """k-means with k clusters on standardised features; per cluster, the member nearest its centroid.
    Distances are computed to each row's own centroid only (O(n*features)), never as an n x k matrix."""
    Xs = StandardScaler().fit_transform(np.asarray(X_pool, dtype=np.float64))
    k = min(k, len(Xs))
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=4096, n_init=1, max_iter=50).fit(Xs)
    labels = km.labels_
    own = np.einsum('ij,ij->i', Xs - km.cluster_centers_[labels], Xs - km.cluster_centers_[labels])
    df = pd.DataFrame({'lab': labels, 'd': own})
    chosen = df.groupby('lab').d.idxmin().values.astype(int)
    if len(chosen) < k:
        rest = np.setdiff1d(np.arange(len(Xs)), chosen)
        chosen = np.concatenate([chosen, rest[np.argsort(own[rest])[:k - len(chosen)]]])
    return chosen

def acquire_hybrid(X_pool, p_pool, k, seed, factor=5):
    cand = acquire_uncertainty(p_pool, min(len(p_pool), factor * k))
    sub = acquire_diversity(np.asarray(X_pool)[cand], k, seed)
    return cand[sub]

# ---------- cross-validated buffer-only estimate on the buffer itself ----------
def cv_estimate(make_model_fn, Xb, yb, seed, n_splits=3):
    """Mean MCC over stratified folds; returns 0.0 when a class has fewer than n_splits rows."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2 or np.bincount(yb).min() < n_splits:
        return 0.0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for tr, te in skf.split(Xb, yb):
        if len(np.unique(yb[tr])) < 2:
            out.append(0.0); continue
        mdl = make_model_fn(len(tr)); mdl.fit(Xb.iloc[tr], yb[tr])
        out.append(mcc_of_pred(yb[te], (mdl.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
    return float(np.mean(out))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
from sklearn.model_selection import train_test_split

def record(row):
    r = {c: row.get(c, np.nan) for c in COLS}
    pd.DataFrame([r], columns=COLS).to_csv(V3_CSV, mode='a', index=False, header=not os.path.exists(V3_CSV))

def key(seed, src, tgt, m, b, s):
    return (str(seed), src, tgt, m, f'{float(b):.6g}', s)

done = set()
if os.path.exists(V3_CSV):
    prev = pd.read_csv(V3_CSV)
    done = set(key(r.seed, r.source, r.target, r.model, r.budget, r.strategy) for r in prev.itertuples())
    print(f'resume: {len(done)} rows already recorded')

def is_done(*k):
    return key(*k) in done

def mark(seed, src, tgt, m, b, s, metrics, n_train, **extra):
    row = dict(seed=seed, source=src, target=tgt, model=m, budget=b, strategy=s, n_train=n_train, **metrics, **extra)
    record(row)
    done.add(key(seed, src, tgt, m, b, s))
    print(f"  s{seed} {src}->{tgt} {m} b={b} {s}: MCC={metrics['mcc']:.3f}")

def fit_eval_buffer(mname, seed, Xb, yb, Xev, yev):
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2:
        m = all_metrics(yev, np.full(len(yev), float(yb[0]))); m['mcc'] = 0.0
        return m, 0
    model = make_model(mname, seed, len(Xb))
    t0 = time.time(); model.fit(Xb, yb)
    m = all_metrics(yev, model.predict_proba(Xev)[:, 1])
    fs = round(time.time() - t0); del model; gc.collect()
    return m, fs

def budget_rows(n_full, b):
    return max(1, int(round(n_full * b)))

ACQ_SHARED = ['uniform', 'diversity']
ACQ_SOURCE = ['uncertainty', 'hybrid']

for seed in CFG['seeds']:
    parts = {}
    for tag, d in DATASETS.items():
        tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
        tr = tr.reset_index(drop=True)
        parts[tag] = dict(train_full=tr, train=stratified_cap(tr, CFG['train_cap'], seed),
                          eval=stratified_cap(te, CFG['eval_cap'], seed), pool=stratified_cap(tr, CFG['pool_cap'], seed))

    for tgt in CFG['corpora']:
        pool = parts[tgt]['pool']; ev = parts[tgt]['eval']; yev = ev['Label'].values
        need = any(not is_done(seed, 'any', tgt, m, b, f'acq_{a}') for m in MODELS for b in CFG['acq_budgets'] for a in ACQ_SHARED)
        if not need:
            continue
        Xpool_raw, _ = clean_X(pool, FEATURES)
        for b in CFG['acq_budgets']:
            k = budget_rows(len(parts[tgt]['train_full']), b)
            sels = {'uniform': acquire_uniform(len(pool), k, seed), 'diversity': acquire_diversity(Xpool_raw.values, k, seed)}
            for a in ACQ_SHARED:
                buf = pool.iloc[sels[a]]
                Xb, mb = clean_X(buf, FEATURES); Xev_b, _ = clean_X(ev, FEATURES, medians=mb)
                for mname in MODELS:
                    if is_done(seed, 'any', tgt, mname, b, f'acq_{a}'):
                        continue
                    m, fs = fit_eval_buffer(mname, seed, Xb, buf['Label'].values, Xev_b, yev)
                    mark(seed, 'any', tgt, mname, b, f'acq_{a}', m, len(buf), n_benign_buf=int((buf['Label'] == 0).sum()), fit_s=fs)
                del Xb, Xev_b
        del Xpool_raw; gc.collect()

    for src in CFG['corpora']:
        others = [t for t in CFG['corpora'] if t != src]
        Xs, med_s = clean_X(parts[src]['train'], FEATURES); ys = parts[src]['train']['Label'].values
        for mname in MODELS:
            need = any(not is_done(seed, src, t, mname, 0, 'zero_shot_2s') for t in others) or any(
                not is_done(seed, src, t, mname, b, s) for t in others for b in CFG['budgets'] for s in ('rethreshold_2s', 'selector')) or any(
                not is_done(seed, src, t, mname, b, f'acq_{a}') for t in others for b in CFG['acq_budgets'] for a in ACQ_SOURCE)
            if not need:
                continue
            model = make_model(mname, seed, len(Xs)); t0 = time.time(); model.fit(Xs, ys)
            print(f'seed {seed} | source fit {mname} on {src}: {time.time()-t0:.0f}s')
            for tgt in others:
                ev = parts[tgt]['eval']; yev = ev['Label'].values
                Xev, _ = clean_X(ev, FEATURES, medians=med_s); p = model.predict_proba(Xev)[:, 1]; del Xev
                if not is_done(seed, src, tgt, mname, 0, 'zero_shot_2s'):
                    m2, t2, o2 = best_threshold_two_sided(yev, p)
                    m = all_metrics(yev, p); m['mcc_best_thr'] = m2; m['thr_best'] = t2
                    mark(seed, src, tgt, mname, 0, 'zero_shot_2s', m, len(Xs), orient=o2)
                for b in CFG['budgets']:
                    if is_done(seed, src, tgt, mname, b, 'rethreshold_2s') and is_done(seed, src, tgt, mname, b, 'selector'):
                        continue
                    buf = stratified_frac(parts[tgt]['train_full'], b, seed); ybuf = buf['Label'].values
                    Xb, _ = clean_X(buf, FEATURES, medians=med_s); pb = model.predict_proba(Xb)[:, 1]; del Xb
                    bm, bt, bo = best_threshold_two_sided(ybuf, pb)
                    if not is_done(seed, src, tgt, mname, b, 'rethreshold_2s'):
                        m = all_metrics(yev, p); m['mcc'] = mcc_of_pred(yev, apply_oriented(p, bt, bo))
                        mark(seed, src, tgt, mname, b, 'rethreshold_2s', m, len(buf), orient=bo, buffer_est=bm, n_benign_buf=int((ybuf == 0).sum()))
                    if not is_done(seed, src, tgt, mname, b, 'selector'):
                        Xbf, _ = clean_X(buf, FEATURES)
                        cv = cv_estimate(lambda n: make_model(mname, seed, n), Xbf, ybuf, seed, CFG['cv_folds'])
                        m = all_metrics(yev, p); m['mcc'] = np.nan
                        mark(seed, src, tgt, mname, b, 'selector', m, len(buf), orient=int(bm > cv), buffer_est=bm, cv_est=cv, n_benign_buf=int((ybuf == 0).sum()))
                        del Xbf
                if any(not is_done(seed, src, tgt, mname, b, f'acq_{a}') for b in CFG['acq_budgets'] for a in ACQ_SOURCE):
                    pool = parts[tgt]['pool']
                    Xpool, _ = clean_X(pool, FEATURES, medians=med_s); ppool = model.predict_proba(Xpool)[:, 1]; del Xpool
                    Xpool_raw, _ = clean_X(pool, FEATURES)
                    for b in CFG['acq_budgets']:
                        k = budget_rows(len(parts[tgt]['train_full']), b)
                        for a in ACQ_SOURCE:
                            if is_done(seed, src, tgt, mname, b, f'acq_{a}'):
                                continue
                            sel = acquire_uncertainty(ppool, k) if a == 'uncertainty' else acquire_hybrid(Xpool_raw.values, ppool, k, seed)
                            bufa = pool.iloc[sel]
                            Xb, mb = clean_X(bufa, FEATURES); Xev_b, _ = clean_X(ev, FEATURES, medians=mb)
                            m, fs = fit_eval_buffer(mname, seed, Xb, bufa['Label'].values, Xev_b, yev)
                            mark(seed, src, tgt, mname, b, f'acq_{a}', m, len(bufa), n_benign_buf=int((bufa['Label'] == 0).sum()), fit_s=fs)
                            del Xb, Xev_b
                    del Xpool_raw; gc.collect()
            del model; gc.collect()
        del Xs; gc.collect()
print('rows recorded:', len(done))

In [ ]:
from scipy.stats import wilcoxon

v3 = pd.read_csv(V3_CSV).drop_duplicates(['seed', 'source', 'target', 'model', 'budget', 'strategy'])
v1 = pd.read_csv(f'{RESULT}/fc_results.csv').drop_duplicates(['seed', 'source', 'target', 'model', 'budget', 'strategy'])
v2 = pd.read_csv(f'{RESULT}/fc_results_v2.csv').drop_duplicates(['seed', 'source', 'target', 'model', 'budget', 'strategy'])
K = ['seed', 'source', 'target', 'model', 'budget']

z2 = v3[v3.strategy == 'zero_shot_2s'][['seed', 'source', 'target', 'model', 'mcc_best_thr', 'orient']].rename(columns={'mcc_best_thr': 'ceiling_2s'})
z1 = v2[v2.strategy == 'zero_shot'][['seed', 'source', 'target', 'model', 'mcc_best_thr']].rename(columns={'mcc_best_thr': 'ceiling_1s'})
zc = z2.merge(z1, on=['seed', 'source', 'target', 'model'])
print(f'ceiling: one-sided mean {zc.ceiling_1s.mean():.3f} | two-sided mean {zc.ceiling_2s.mean():.3f} | inverted cells {int((zc.orient == -1).sum())}/{len(zc)}')
r2 = v3[v3.strategy == 'rethreshold_2s'][K + ['mcc', 'buffer_est']].rename(columns={'mcc': 'rt2_mcc'})
r1 = v2[v2.strategy == 'rethreshold'][K + ['mcc']].rename(columns={'mcc': 'rt1_mcc'})
bo = v1[v1.strategy == 'buffer_only'][K + ['mcc']].rename(columns={'mcc': 'bo_mcc'})
d = r2.merge(r1, on=K).merge(bo, on=K).merge(zc[['seed', 'source', 'target', 'model', 'ceiling_2s']], on=['seed', 'source', 'target', 'model'])
print('\nper budget: ceiling_2s | rethreshold one-sided | rethreshold two-sided | buffer_only | cells where ceiling_2s > buffer_only')
print(d.groupby('budget').agg(ceiling_2s=('ceiling_2s', 'mean'), rt_1s=('rt1_mcc', 'mean'), rt_2s=('rt2_mcc', 'mean'), buffer_only=('bo_mcc', 'mean'),
      ceiling_wins=('ceiling_2s', lambda s: int((s > d.loc[s.index, 'bo_mcc']).sum()))).round(3))
print(f'bound violations (rethreshold_2s > ceiling_2s): {int((d.rt2_mcc > d.ceiling_2s + 1e-6).sum())}')
d.to_csv(f'{RESULT}/fc_diagnostic.csv', index=False)

sel = v3[v3.strategy == 'selector'][K + ['orient', 'buffer_est', 'cv_est']].rename(columns={'orient': 'chose_rt'})
s = sel.merge(d[K + ['rt2_mcc', 'bo_mcc']], on=K)
s['realised'] = np.where(s.chose_rt == 1, s.rt2_mcc, s.bo_mcc)
s['oracle'] = np.maximum(s.rt2_mcc, s.bo_mcc)
s['regret'] = s.oracle - s.realised
s['always_retrain_regret'] = s.oracle - s.bo_mcc
s['always_rt_regret'] = s.oracle - s.rt2_mcc
s['correct'] = (s.chose_rt == 1) == (s.rt2_mcc >= s.bo_mcc)
print('\nselector per budget: chose_rethreshold rate | mean regret (selector / always-retrain / always-rethreshold) | decision accuracy')
print(s.groupby('budget').agg(chose_rt=('chose_rt', 'mean'), regret=('regret', 'mean'), always_retrain=('always_retrain_regret', 'mean'),
      always_rt=('always_rt_regret', 'mean'), accuracy=('correct', 'mean')).round(3))
s.to_csv(f'{RESULT}/fc_selector.csv', index=False)

base = v1[(v1.strategy == 'buffer_only') & (v1.budget.isin(CFG['acq_budgets']))].groupby(['seed', 'target', 'model', 'budget']).mcc.mean().rename('random_strat').reset_index()
acq = v3[v3.strategy.str.startswith('acq_')].copy(); acq['rule'] = acq.strategy.str.replace('acq_', '')
shared = acq[acq.source == 'any'].groupby(['seed', 'target', 'model', 'budget', 'rule']).mcc.mean().reset_index()
srcdep = acq[acq.source != 'any'].groupby(['seed', 'target', 'model', 'budget', 'rule']).mcc.mean().reset_index()
a = pd.concat([shared, srcdep]).merge(base, on=['seed', 'target', 'model', 'budget'])
a['gain'] = a.mcc - a.random_strat
rows = []
for (b, rule), g in a.groupby(['budget', 'rule']):
    p = wilcoxon(g.mcc, g.random_strat).pvalue if len(g) > 5 and (g.gain != 0).any() else np.nan
    rows.append(dict(budget=b, rule=rule, n=len(g), rule_mcc=g.mcc.mean(), random_strat=g.random_strat.mean(), gain=g.gain.mean(),
                     frac_better=(g.gain > 0).mean(), wilcoxon_p=p))
acq_tab = pd.DataFrame(rows).round(4)
print('\nacquisition vs stratified-random baseline:'); print(acq_tab.to_string(index=False))
acq_tab.to_csv(f'{RESULT}/fc_acquisition.csv', index=False)
print('\nbenign rows per acquired buffer (mean), rule x budget:')
print(acq.groupby(['budget', 'rule']).n_benign_buf.mean().round(1).unstack().to_string())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "12: oriented two-sided ceiling, buffer acquisition rules (uniform/uncertainty/diversity/hybrid), ceiling-gated selector; selector and acquisition tables"],
  capture_output=True, text=True)
print(r.stdout)
print(r.stderr)